# 01 - Tiền xử lý dữ liệu cổ phiếu và tin tức



## Mục tiêu

1. Đọc đúng hai file Excel và đúng sheet dữ liệu.
2. Giữ nguyên dữ liệu thô, chỉ xử lý trên bản sao.
3. Chuẩn hóa tên cột, kiểu dữ liệu, mã cổ phiếu và thời gian.
4. Kiểm tra dữ liệu thiếu, dữ liệu trùng và tính hợp lệ của OHLCV.
5. Phân biệt tin cấp doanh nghiệp và tin cấp ngành (`ALL`).
6. Đánh dấu tin đăng lại nhưng không tự động xóa các bài khác nguồn.
7. Tạo ba bảng kết quả trong bộ nhớ: `stock_clean`, `news_clean`, `news_for_linking`.



## 1. Chuẩn bị dữ liệu đầu vào

In [1]:
from pathlib import Path
import html
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

print('Pandas version:', pd.__version__)

Pandas version: 2.3.0


In [2]:
def find_source_file(pattern: str) -> Path:
    """Tìm file theo nhiều vị trí thường gặp và trả về đường dẫn đầu tiên."""
    cwd = Path.cwd()
    project_root = cwd.parent if cwd.name == 'notebooks' else cwd

    candidate_dirs = [
        project_root / 'data' / 'raw',
        cwd / 'data' / 'raw',
        cwd,
        project_root / 'upload',
        cwd / 'upload',
    ]

    for directory in candidate_dirs:
        if directory.exists():
            matches = sorted(directory.glob(pattern))
            if matches:
                return matches[0].resolve()

    searched = '\n'.join(str(path.resolve()) for path in candidate_dirs)
    raise FileNotFoundError(
        f'Không tìm thấy file có mẫu {pattern}. Đã tìm tại:\n{searched}'
    )


STOCK_PATH = find_source_file('COPHIEU_BANLE_TIEUDUNG*.xlsx')
NEWS_PATH = find_source_file('K4_CRAWL_NHOM_4_BAN_LE_TIEU_DUNG_DUOC*.xlsx')

STOCK_SHEET = 'CRAWL_STOCK_DNSE'
NEWS_SHEET = 'NEWS_RAW'

print('File cổ phiếu:', STOCK_PATH)
print('File tin tức  :', NEWS_PATH)

File cổ phiếu: D:\KLTN\stock_anomaly_detection\data\raw\COPHIEU_BANLE_TIEUDUNG.xlsx
File tin tức  : D:\KLTN\stock_anomaly_detection\data\raw\K4_CRAWL_NHOM_4_BAN_LE_TIEU_DUNG_DUOC.xlsx


In [3]:
# Đọc dữ liệu và giữ nguyên hai bảng raw để đối chiếu khi cần.
stock_raw = pd.read_excel(STOCK_PATH, sheet_name=STOCK_SHEET)
news_raw = pd.read_excel(NEWS_PATH, sheet_name=NEWS_SHEET)

print('Kích thước stock_raw:', stock_raw.shape)
print('Kích thước news_raw :', news_raw.shape)

display(stock_raw.head(3))
display(news_raw.head(3))

Kích thước stock_raw: (4690, 8)
Kích thước news_raw : (1172, 18)


,Ticker,Date,Open,High,Low,Close,Volume,Crawl Timestamp
0,MWG,2026-08-05,72.4,72.9,71.0,72.2,3955700,2026-09-04T08:48:45.157Z
1,MWG,2026-08-06,71.5,71.9,69.8,71.2,5330100,2026-09-04T08:48:45.157Z
2,MWG,2026-08-07,71.3,71.6,70.2,71.0,3084600,2026-09-04T08:48:45.157Z


,news_id,title,summary,content,published_date,source,url,industry_group,tickers,keywords,event_type,crawl_time,content_hash,crawl_status,error_message,checked_by,checked_time,note
0,RSS:CAFE_20260609_2D8FEE,"CEO của VNG nhận lương gần 1,3 tỷ đồng/tháng, vẫn chỉ bằng hơn một nửa CEO Masan","VNZ&MSN: Giá hiện tại Thay đổi Xem hồ sơ doanh nghiệp TIN MỚI Trên thị trường, có 2 CEO trẻ nổi ...","VNZ&MSN: Giá hiện tại Thay đổi Xem hồ sơ doanh nghiệp TIN MỚI Trên thị trường, có 2 CEO trẻ nổi ...",2026-06-09 00:09:00,RSS:Cafef DN,https://cafef.vn/ceo-cua-vng-nhan-luong-gan-13-ty-dong-thang-van-chi-bang-hon-mot-nua-ceo-masan-...,Bán lẻ / Tiêu dùng,MSN,MSN,"stock_mention, company_news",2026-06-10 13:49:41,2D8FEE,success,NaN,NaN,NaN,NaN
1,RSS:VIET_20260602_90FD02,Masan Consumer tham gia Triển lãm thực phẩm và đồ uống quốc tế tại Thái Lan,# Thi THPT # World Cup 2026 Podcast Tuần Việt Nam Chính trị Sự kiện Xây dựng đảng Đối ngoại Bàn ...,# Thi THPT # World Cup 2026 Podcast Tuần Việt Nam Chính trị Sự kiện Xây dựng đảng Đối ngoại Bàn ...,2026-06-02 16:20:00,RSS:Vietnamnet KD,https://vietnamnet.vn/masan-consumer-tham-gia-trien-lam-thuc-pham-va-do-uong-quoc-te-tai-thai-la...,Bán lẻ / Tiêu dùng,MSN,Masan Consumer,"earnings, expansion, contraction, stock_mention",2026-06-10 13:49:42,90FD02,success,NaN,NaN,NaN,NaN
2,RSS:VIET_20260530_54D354,Vinamilk kể câu chuyện ‘hành trình 50 năm’ tại Triển lãm chuyên ngành sữa 2026,# Thi THPT # World Cup 2026 Tuần Việt Nam Chính trị Sự kiện Xây dựng đảng Đối ngoại Bàn luận Kỷ ...,# Thi THPT # World Cup 2026 Tuần Việt Nam Chính trị Sự kiện Xây dựng đảng Đối ngoại Bàn luận Kỷ ...,2026-05-30 14:04:47,RSS:Vietnamnet KD,https://vietnamnet.vn/vinamilk-ke-cau-chuyen-hanh-trinh-50-nam-tai-trien-lam-chuyen-nganh-sua-20...,Bán lẻ / Tiêu dùng,VNM,VNM,"stock_mention, company_news",2026-06-10 13:49:43,54D354,success,NaN,NaN,NaN,NaN


## 2. Tiền xử lý dữ liệu cổ phiếu

| Cột gốc | Cột chuẩn hóa | Xử lý | Mục đích |
|---|---|---|---|
| `Ticker` | `ticker` | Xóa khoảng trắng, chuyển chữ hoa | Chia dữ liệu theo mã |
| `Date` | `date` | Chuyển thành ngày, bỏ phần giờ | Sắp xếp và ghép dữ liệu |
| `Open` | `open` | Chuyển thành số | Giá mở cửa |
| `High` | `high` | Chuyển thành số | Giá cao nhất |
| `Low` | `low` | Chuyển thành số | Giá thấp nhất |
| `Close` | `close` | Chuyển thành số | Giá đóng cửa |
| `Volume` | `volume` | Chuyển thành số nguyên không âm | Khối lượng giao dịch |
| `Crawl Timestamp` | `crawl_timestamp` | Chuyển thành thời gian UTC | Theo dõi thời điểm thu thập, không đưa vào mô hình |

Một dòng OHLCV hợp lệ phải thỏa mãn: giá dương, khối lượng không âm, `High` không thấp hơn `Open/Close`, `Low` không cao hơn `Open/Close`, và `High >= Low`. Dòng sai được giữ riêng trong `stock_invalid` để kiểm tra, không tự ý sửa giá.

In [4]:
STOCK_RENAME = {
    'Ticker': 'ticker',
    'Date': 'date',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'Crawl Timestamp': 'crawl_timestamp',
}

REQUIRED_STOCK_COLUMNS = [
    'ticker', 'date', 'open', 'high', 'low', 'close', 'volume'
]


def preprocess_stock(stock_input: pd.DataFrame):
    stock = stock_input.copy()
    stock.columns = stock.columns.astype(str).str.strip()

    missing_source_columns = set(STOCK_RENAME) - set(stock.columns)
    if missing_source_columns:
        raise ValueError(f'Thiếu cột cổ phiếu: {sorted(missing_source_columns)}')

    stock = stock.rename(columns=STOCK_RENAME)

    # Chuẩn hóa mã và ngày.
    stock['ticker'] = stock['ticker'].astype('string').str.strip().str.upper()
    stock['date'] = pd.to_datetime(stock['date'], errors='coerce').dt.normalize()
    stock['crawl_timestamp'] = pd.to_datetime(
        stock['crawl_timestamp'], errors='coerce', utc=True
    )

    # Chuyển OHLCV thành dữ liệu số. Ký tự sai sẽ thành NaN để kiểm tra.
    numeric_columns = ['open', 'high', 'low', 'close', 'volume']
    for column in numeric_columns:
        stock[column] = pd.to_numeric(stock[column], errors='coerce')

    missing_required_mask = stock[REQUIRED_STOCK_COLUMNS].isna().any(axis=1)
    stock_missing_required = stock.loc[missing_required_mask].copy()
    stock = stock.loc[~missing_required_mask].copy()

    # Nếu crawl nhiều lần cùng mã-ngày, giữ lần crawl mới nhất.
    duplicate_rows = int(stock.duplicated(['ticker', 'date'], keep=False).sum())
    stock = (
        stock.sort_values(['ticker', 'date', 'crawl_timestamp'])
        .drop_duplicates(['ticker', 'date'], keep='last')
        .reset_index(drop=True)
    )

    positive_price = stock[['open', 'high', 'low', 'close']].gt(0).all(axis=1)
    valid_volume = stock['volume'].ge(0)
    valid_high = stock['high'].ge(stock[['open', 'close']].max(axis=1))
    valid_low = stock['low'].le(stock[['open', 'close']].min(axis=1))
    valid_range = stock['high'].ge(stock['low'])

    stock['ohlcv_valid'] = (
        positive_price & valid_volume & valid_high & valid_low & valid_range
    )

    stock_invalid = stock.loc[~stock['ohlcv_valid']].copy()
    stock_clean = stock.loc[stock['ohlcv_valid']].copy()
    stock_clean['volume'] = stock_clean['volume'].round().astype('int64')
    stock_clean = (
        stock_clean.drop(columns=['ohlcv_valid'])
        .sort_values(['ticker', 'date'])
        .reset_index(drop=True)
    )

    report = {
        'raw_rows': len(stock_input),
        'missing_required_rows': len(stock_missing_required),
        'duplicate_rows_before_dedup': duplicate_rows,
        'invalid_ohlcv_rows': len(stock_invalid),
        'clean_rows': len(stock_clean),
        'ticker_count': stock_clean['ticker'].nunique(),
        'start_date': stock_clean['date'].min(),
        'end_date': stock_clean['date'].max(),
    }

    return stock_clean, stock_missing_required, stock_invalid, report


stock_clean, stock_missing_required, stock_invalid, stock_report = (
    preprocess_stock(stock_raw)
)

In [5]:
print('BÁO CÁO CHẤT LƯỢNG DỮ LIỆU CỔ PHIẾU')
display(pd.Series(stock_report, name='Giá trị').to_frame())

rows_by_ticker = (
    stock_clean.groupby('ticker')
    .agg(
        rows=('date', 'size'),
        start_date=('date', 'min'),
        end_date=('date', 'max'),
    )
)
display(rows_by_ticker)
display(stock_clean.head())

BÁO CÁO CHẤT LƯỢNG DỮ LIỆU CỔ PHIẾU


,Giá trị
raw_rows,4690
missing_required_rows,0
duplicate_rows_before_dedup,0
invalid_ohlcv_rows,0
clean_rows,4690
ticker_count,14
start_date,2025-05-05 00:00:00
end_date,2026-09-04 00:00:00


,rows,start_date,end_date
ticker,,,
ANV,335,2025-05-05,2026-09-04
DBC,335,2025-05-05,2026-09-04
DGW,335,2025-05-05,2026-09-04
DHG,335,2025-05-05,2026-09-04
FRT,335,2025-05-05,2026-09-04
IMP,335,2025-05-05,2026-09-04
MSN,335,2025-05-05,2026-09-04
MWG,335,2025-05-05,2026-09-04
PNJ,335,2025-05-05,2026-09-04


,ticker,date,open,high,low,close,volume,crawl_timestamp
0,ANV,2025-05-05,14.76,14.81,14.76,14.81,426000,2026-09-04 09:15:08.321000+00:00
1,ANV,2025-05-06,13.99,15.20,13.79,15.20,3802000,2026-09-04 09:15:08.321000+00:00
2,ANV,2025-05-07,15.10,15.30,14.76,15.25,686800,2026-09-04 09:15:08.321000+00:00
3,ANV,2025-05-08,15.25,15.25,14.76,15.10,1323000,2026-09-04 09:15:08.321000+00:00
4,ANV,2025-05-09,15.00,15.00,14.09,14.62,1977000,2026-09-04 09:15:08.321000+00:00


In [6]:
# Kiểm tra số phiên giao dịch giữa các mã
phiên_counts = stock_clean.groupby('ticker').size().sort_values(ascending=False)
print("Số phiên giao dịch theo mã:")
display(phiên_counts.to_frame('Số phiên'))

# So sánh với mã trung bình
mean_phiên = phiên_counts.mean()
std_phiên = phiên_counts.std()
print(f"Trung bình: {mean_phiên:.1f} phiên, Độ lệch chuẩn: {std_phiên:.1f}")

# Cảnh báo nếu có mã lệch quá nhiều (chênh > 2*std)
if std_phiên > 5:
    print("Có sự chênh lệch đáng kể về số phiên giữa các mã. Cần kiểm tra các mã sau:")
    print(phiên_counts[phiên_counts < mean_phiên - 2*std_phiên])

Số phiên giao dịch theo mã:


,Số phiên
ticker,
ANV,335
DBC,335
DGW,335
DHG,335
FRT,335
IMP,335
MSN,335
MWG,335
PNJ,335


Trung bình: 335.0 phiên, Độ lệch chuẩn: 0.0


## 3. Tiền xử lý dữ liệu tin tức

### Quy tắc theo từng cột

| Cột | Cách xử lý | Vai trò |
|---|---|---|
| `news_id` | Làm sạch chuỗi, kiểm tra trùng | Định danh bài |
| `title` | Xóa HTML, chuẩn hóa khoảng trắng | Văn bản chính |
| `summary` | Làm sạch; thiếu thì tạo cờ và thay bằng chuỗi rỗng | Văn bản phân tích |
| `content` | Làm sạch; thiếu thì tạo cờ và thay bằng chuỗi rỗng | Nội dung bổ sung |
| `published_date` | Chuyển thành thời gian và giữ giờ đăng | Ghép với phiên giao dịch sau này |
| `source` | Làm sạch chuỗi | Phân tích nguồn báo |
| `url` | Làm sạch và kiểm tra trùng | Nhận biết bản crawl trùng |
| `industry_group` | Giữ bản gốc và tạo nhãn chuẩn hóa | Thống kê nhóm ngành |
| `tickers` | Giữ bản gốc và tách thành danh sách mã | Liên kết doanh nghiệp |
| `keywords` | Làm sạch chuỗi | Phân tích từ khóa sau này |
| `event_type` | Làm sạch, chưa tách multi-label | Phân tích sự kiện sau này |
| `crawl_time` | Chuyển thành thời gian | Kiểm tra độ trễ thu thập |
| `content_hash` | Làm sạch | Kiểm tra nội dung trùng |
| `crawl_status` | Chuyển chữ thường | Theo dõi cách bài được thu thập |
| `error_message`, `checked_by`, `checked_time`, `note` | Chỉ bỏ nếu trống toàn bộ | Cột quản trị |

> Không lọc riêng `crawl_status == 'success'`. Các trạng thái `keyword_matched` và `stock_detected` vẫn là bài hợp lệ.

In [7]:
def clean_text(value):
    """Xóa HTML và khoảng trắng thừa nhưng giữ nguyên tiếng Việt."""
    if pd.isna(value):
        return pd.NA

    value = html.unescape(str(value))
    value = re.sub(r'<[^>]+>', ' ', value)
    value = re.sub(r'\s+', ' ', value).strip()
    return value if value else pd.NA


def parse_tickers(value):
    """Ví dụ 'MWG, MSN' -> ['MSN', 'MWG']; vẫn giữ mã ALL."""
    if pd.isna(value):
        return []

    tokens = re.split(r'[,;/|\s]+', str(value).upper().strip())
    return sorted(set(token.strip() for token in tokens if token.strip()))


def normalize_title(value):
    """Tạo khóa tiêu đề để đánh dấu các bài có thể là tin đăng lại."""
    if pd.isna(value):
        return ''

    value = unicodedata.normalize('NFKC', str(value)).lower()
    value = re.sub(r'[^\w\s]', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()


def classify_news_level(company_tickers, ticker_list):
    if company_tickers:
        return 'company'
    if 'ALL' in ticker_list:
        return 'industry'
    return 'invalid'

In [8]:
REQUIRED_NEWS_COLUMNS = [
    'news_id', 'title', 'published_date', 'source', 'url', 'tickers'
]

TEXT_COLUMNS = [
    'news_id', 'title', 'summary', 'content', 'source', 'url',
    'industry_group', 'tickers', 'keywords', 'event_type',
    'content_hash', 'crawl_status'
]

ADMIN_COLUMNS = ['error_message', 'checked_by', 'checked_time', 'note']

INDUSTRY_MAPPING = {
    'Bán lẻ / Tiêu dùng': 'Bán lẻ - Tiêu dùng - Thực phẩm - Dược phẩm',
    'Bán lẻ / FMCG': 'Bán lẻ - Tiêu dùng - Thực phẩm - Dược phẩm',
}


def preprocess_news(news_input: pd.DataFrame, valid_tickers):
    news = news_input.copy()
    news.columns = news.columns.astype(str).str.strip()

    missing_source_columns = set(REQUIRED_NEWS_COLUMNS) - set(news.columns)
    if missing_source_columns:
        raise ValueError(f'Thiếu cột tin tức: {sorted(missing_source_columns)}')

    for column in TEXT_COLUMNS:
        if column in news.columns:
            news[column] = news[column].map(clean_text)

    news['crawl_status'] = (
        news['crawl_status'].astype('string').str.strip().str.lower()
    )
    news['published_date'] = pd.to_datetime(news['published_date'], errors='coerce')
    news['crawl_time'] = pd.to_datetime(news['crawl_time'], errors='coerce')
    news['published_day'] = news['published_date'].dt.normalize()

    # Ghi nhận missing trước khi điền chuỗi rỗng.
    news['summary_missing'] = news['summary'].isna()
    news['content_missing'] = news['content'].isna()
    news['summary'] = news['summary'].fillna('')
    news['content'] = news['content'].fillna('')

    missing_required_mask = news[REQUIRED_NEWS_COLUMNS].isna().any(axis=1)
    news_missing_required = news.loc[missing_required_mask].copy()
    news = news.loc[~missing_required_mask].copy()

    # Chỉ xóa bản crawl kỹ thuật bị trùng news_id hoặc URL.
    duplicate_news_id_rows = int(news.duplicated('news_id', keep=False).sum())
    duplicate_url_rows = int(news.duplicated('url', keep=False).sum())
    duplicate_hash_rows = int(news.duplicated('content_hash', keep=False).sum())

    news = (
        news.sort_values('crawl_time')
        .drop_duplicates('news_id', keep='last')
        .drop_duplicates('url', keep='last')
        .copy()
    )

    # Tạo danh sách ticker và kiểm tra với 14 mã có dữ liệu giá.
    news['ticker_list'] = news['tickers'].map(parse_tickers)
    news['company_tickers'] = news['ticker_list'].map(
        lambda values: [ticker for ticker in values if ticker in valid_tickers]
    )
    news['invalid_tickers'] = news['ticker_list'].map(
        lambda values: [
            ticker for ticker in values
            if ticker not in valid_tickers and ticker != 'ALL'
        ]
    )
    news['news_level'] = news.apply(
        lambda row: classify_news_level(row['company_tickers'], row['ticker_list']),
        axis=1,
    )
    news['multi_ticker_flag'] = news['company_tickers'].str.len().gt(1)

    # Giữ cả nhãn ngành gốc và nhãn đã thống nhất.
    news['industry_group_original'] = news['industry_group']
    news['industry_group_std'] = news['industry_group'].replace(INDUSTRY_MAPPING)

    # Chỉ đánh dấu tin đăng lại, không tự động xóa bài khác nguồn.
    news['title_key'] = news['title'].map(normalize_title)
    news['story_duplicate_flag'] = news.duplicated(
        ['title_key', 'published_day'], keep=False
    )

    # title + summary ổn định hơn content vì content khác nhau nhiều giữa nguồn RSS/API.
    news['text_for_analysis'] = (
        news['title'].fillna('') + '. ' + news['summary'].fillna('')
    ).str.strip()

    # Chỉ loại cột quản trị khi toàn bộ cột đều trống.
    empty_admin_columns = [
        column for column in ADMIN_COLUMNS
        if column in news.columns and news[column].isna().all()
    ]
    news = news.drop(columns=empty_admin_columns)

    news_clean = (
        news.sort_values(['published_date', 'news_id'])
        .reset_index(drop=True)
    )

    report = {
        'raw_rows': len(news_input),
        'missing_required_rows': len(news_missing_required),
        'duplicate_news_id_rows': duplicate_news_id_rows,
        'duplicate_url_rows': duplicate_url_rows,
        'duplicate_hash_rows': duplicate_hash_rows,
        'summary_missing_rows': int(news_clean['summary_missing'].sum()),
        'content_missing_rows': int(news_clean['content_missing'].sum()),
        'possible_reposted_rows': int(news_clean['story_duplicate_flag'].sum()),
        'multi_ticker_rows': int(news_clean['multi_ticker_flag'].sum()),
        'invalid_ticker_rows': int(news_clean['invalid_tickers'].str.len().gt(0).sum()),
        'company_articles': int(news_clean['news_level'].eq('company').sum()),
        'industry_articles': int(news_clean['news_level'].eq('industry').sum()),
        'clean_rows': len(news_clean),
        'start_date': news_clean['published_date'].min(),
        'end_date': news_clean['published_date'].max(),
        'dropped_empty_admin_columns': ', '.join(empty_admin_columns),
    }

    return news_clean, news_missing_required, report


valid_stock_tickers = set(stock_clean['ticker'].unique())
news_clean, news_missing_required, news_report = preprocess_news(
    news_raw, valid_stock_tickers
)

In [9]:
print('BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TIN TỨC')
display(pd.Series(news_report, name='Giá trị').to_frame())

print('Phân bố cấp độ tin:')
display(news_clean['news_level'].value_counts().rename_axis('news_level').to_frame('articles'))

print('Các mã được nhắc đến nhiều nhất:')
ticker_mentions = (
    news_clean[['news_id', 'company_tickers']]
    .explode('company_tickers')
    .dropna(subset=['company_tickers'])
    .groupby('company_tickers')
    .size()
    .sort_values(ascending=False)
    .rename('mentions')
)
display(ticker_mentions.to_frame())
display(news_clean.head(3))

BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TIN TỨC


,Giá trị
raw_rows,1172
missing_required_rows,0
duplicate_news_id_rows,0
duplicate_url_rows,0
duplicate_hash_rows,0
summary_missing_rows,2
content_missing_rows,20
possible_reposted_rows,44
multi_ticker_rows,76
invalid_ticker_rows,0


Phân bố cấp độ tin:


,articles
news_level,
company,713
industry,459


Các mã được nhắc đến nhiều nhất:


,mentions
company_tickers,
MSN,193
PNJ,145
MWG,142
VNM,124
SAB,55
FRT,48
DBC,29
IMP,23
DGW,21


,news_id,title,summary,content,published_date,source,url,industry_group,tickers,keywords,event_type,crawl_time,content_hash,crawl_status,published_day,summary_missing,content_missing,ticker_list,company_tickers,invalid_tickers,news_level,multi_ticker_flag,industry_group_original,industry_group_std,title_key,story_duplicate_flag,text_for_analysis
0,API:CAFE_20250102_143502,"Quyết tâm nâng tầm bộ máy quản trị, TTC AgriS đẩy mạnh đầu tư mở rộng quy mô","TIN MỚI Theo đó, TTC AgriS (Công ty cổ phần Thành Thành Công – Biên Hòa, HOSE: SBT) sẽ trình ĐHĐ...","TIN MỚI Theo đó, TTC AgriS (Công ty cổ phần Thành Thành Công – Biên Hòa, HOSE: SBT) sẽ trình ĐHĐ...",2025-01-02 20:30:00,API:CafeF DN,https://cafef.vn/quyet-tam-nang-tam-bo-may-quan-tri-ttc-agris-day-manh-dau-tu-mo-rong-quy-mo-188...,Bán lẻ / Tiêu dùng,SBT,SBT,"stock_mention, company_news",2026-06-10 13:54:47,143502,success,2025-01-02,False,False,[SBT],[SBT],[],company,False,Bán lẻ / Tiêu dùng,Bán lẻ - Tiêu dùng - Thực phẩm - Dược phẩm,quyết tâm nâng tầm bộ máy quản trị ttc agris đẩy mạnh đầu tư mở rộng quy mô,False,"Quyết tâm nâng tầm bộ máy quản trị, TTC AgriS đẩy mạnh đầu tư mở rộng quy mô. TIN MỚI Theo đó, T..."
1,API:CAFE_20250104_B81C4A,PNJ tự hào đón nhận Thương hiệu vàng từ UBND TP.HCM,TIN MỚI Chương trình bình chọn danh hiệu Thương hiệu Vàng Thành phố Hồ Chí Minh được thực hiện d...,TIN MỚI Chương trình bình chọn danh hiệu Thương hiệu Vàng Thành phố Hồ Chí Minh được thực hiện d...,2025-01-04 19:30:00,API:CafeF DN,https://cafef.vn/pnj-tu-hao-don-nhan-thuong-hieu-vang-tu-ubnd-tphcm-188250104192619428.chn,Bán lẻ / Tiêu dùng,PNJ,PNJ,"stock_mention, company_news",2026-06-10 13:54:46,B81C4A,success,2025-01-04,False,False,[PNJ],[PNJ],[],company,False,Bán lẻ / Tiêu dùng,Bán lẻ - Tiêu dùng - Thực phẩm - Dược phẩm,pnj tự hào đón nhận thương hiệu vàng từ ubnd tp hcm,False,PNJ tự hào đón nhận Thương hiệu vàng từ UBND TP.HCM. TIN MỚI Chương trình bình chọn danh hiệu Th...
2,API:CAFE_20250108_9F33D4,Trang trại 700 tỷ của Vinamilk và dự án 85.000 tỷ của Hòa Phát tại Quảng Ngãi: Tỉnh có chỉ đạo mới,"TIN MỚI Ảnh: Cổng thông tin điện tử tỉnh Quảng Ngãi Chiều 31/12/2024, Chủ tịch UBND tỉnh Nguyễn ...","TIN MỚI Ảnh: Cổng thông tin điện tử tỉnh Quảng Ngãi Chiều 31/12/2024, Chủ tịch UBND tỉnh Nguyễn ...",2025-01-08 09:11:00,API:CafeF DN,https://cafef.vn/trang-trai-700-ty-cua-vinamilk-va-du-an-85000-ty-cua-hoa-phat-tai-quang-ngai-ti...,Bán lẻ / Tiêu dùng,VNM,VNM,"stock_mention, company_news",2026-06-10 13:54:46,9F33D4,success,2025-01-08,False,False,[VNM],[VNM],[],company,False,Bán lẻ / Tiêu dùng,Bán lẻ - Tiêu dùng - Thực phẩm - Dược phẩm,trang trại 700 tỷ của vinamilk và dự án 85 000 tỷ của hòa phát tại quảng ngãi tỉnh có chỉ đạo mới,False,Trang trại 700 tỷ của Vinamilk và dự án 85.000 tỷ của Hòa Phát tại Quảng Ngãi: Tỉnh có chỉ đạo m...


## 4. Lọc khoảng thời gian dùng để liên kết tin tức

Dữ liệu giá bắt đầu ngày 05/05/2025. Tuy nhiên, đề tài giữ tin từ 01/05/2025 để các bài xuất hiện trước phiên đầu tiên vẫn có thể nằm trong cửa sổ `[-5, 0]`. Việc căn chỉnh tin sau 15:00, cuối tuần và ngày nghỉ sẽ được thực hiện ở bước liên kết tin tức, không làm tại đây.

In [10]:
NEWS_START = pd.Timestamp('2025-05-01')
NEWS_END = pd.Timestamp('2026-09-04')

news_for_linking = (
    news_clean.loc[
        news_clean['published_day'].between(NEWS_START, NEWS_END)
    ]
    .copy()
    .reset_index(drop=True)
)

# Bảng con chỉ gồm tin có ít nhất một mã doanh nghiệp hợp lệ.
news_company_for_linking = (
    news_for_linking.loc[news_for_linking['news_level'].eq('company')]
    .copy()
    .reset_index(drop=True)
)

print('Tổng tin sạch                 :', len(news_clean))
print('Tin trong thời gian liên kết  :', len(news_for_linking))
print('Tin doanh nghiệp để liên kết  :', len(news_company_for_linking))
print('Ngày tin sớm nhất trong cửa sổ:', news_for_linking['published_date'].min())
print('Ngày tin muộn nhất            :', news_for_linking['published_date'].max())

Tổng tin sạch                 : 1172
Tin trong thời gian liên kết  : 1061
Tin doanh nghiệp để liên kết  : 602
Ngày tin sớm nhất trong cửa sổ: 2025-05-02 09:00:00
Ngày tin muộn nhất            : 2026-09-04 16:49:00


## 5. Kiểm tra chất lượng cuối cùng

Các lệnh `assert` giúp dừng pipeline nếu dữ liệu sau làm sạch không đáp ứng yêu cầu. Đây là cách tránh đưa dữ liệu lỗi sang bước tạo đặc trưng.

In [11]:
# Kiểm tra bảng cổ phiếu.
assert not stock_clean[REQUIRED_STOCK_COLUMNS].isna().any().any(), (
    'stock_clean vẫn còn thiếu dữ liệu bắt buộc'
)
assert stock_clean.duplicated(['ticker', 'date']).sum() == 0, (
    'stock_clean vẫn còn trùng ticker-date'
)
assert stock_clean['ticker'].nunique() == 14, (
    'Số mã cổ phiếu sau làm sạch không bằng 14'
)
assert len(stock_invalid) == 0, 'Dữ liệu còn dòng OHLCV không hợp lệ'

# Kiểm tra bảng tin tức.
assert not news_clean[REQUIRED_NEWS_COLUMNS].isna().any().any(), (
    'news_clean vẫn còn thiếu dữ liệu bắt buộc'
)
assert news_clean.duplicated('news_id').sum() == 0, 'news_id vẫn còn trùng'
assert news_clean.duplicated('url').sum() == 0, 'URL vẫn còn trùng'
assert news_clean['invalid_tickers'].str.len().gt(0).sum() == 0, (
    'Vẫn còn ticker không hợp lệ'
)
assert set(news_clean['news_level'].unique()).issubset(
    {'company', 'industry'}
), 'Xuất hiện cấp độ tin không hợp lệ'

print('Tất cả kiểm tra chất lượng cuối cùng đều đạt.')

Tất cả kiểm tra chất lượng cuối cùng đều đạt.


In [12]:
quality_summary = pd.DataFrame({
    'Dataset': ['Cổ phiếu', 'Tin tức', 'Tin trong cửa sổ liên kết'],
    'Số dòng': [len(stock_clean), len(news_clean), len(news_for_linking)],
    'Ngày bắt đầu': [
        stock_clean['date'].min(),
        news_clean['published_date'].min(),
        news_for_linking['published_date'].min(),
    ],
    'Ngày kết thúc': [
        stock_clean['date'].max(),
        news_clean['published_date'].max(),
        news_for_linking['published_date'].max(),
    ],
})

display(quality_summary)

,Dataset,Số dòng,Ngày bắt đầu,Ngày kết thúc
0,Cổ phiếu,4690,2025-05-05 00:00:00,2026-09-04 00:00:00
1,Tin tức,1172,2025-01-02 20:30:00,2026-09-04 16:49:00
2,Tin trong cửa sổ liên kết,1061,2025-05-02 09:00:00,2026-09-04 16:49:00


## 6. Giải thích các bảng kết quả

- `stock_raw`: dữ liệu cổ phiếu gốc, không chỉnh sửa.
- `news_raw`: dữ liệu tin tức gốc, không chỉnh sửa.
- `stock_clean`: 14 mã cổ phiếu đã chuẩn hóa, sắp xếp và kiểm tra OHLCV.
- `stock_missing_required`: các dòng giá thiếu cột bắt buộc để kiểm tra.
- `stock_invalid`: các dòng vi phạm điều kiện OHLCV để kiểm tra.
- `news_clean`: toàn bộ tin hợp lệ sau làm sạch, gồm cả tin công ty và tin ngành.
- `news_missing_required`: các bài thiếu ID, tiêu đề, ngày, nguồn, URL hoặc ticker.
- `news_for_linking`: tin trong khoảng 01/05/2025 đến 04/09/2026.
- `news_company_for_linking`: chỉ các bài có mã doanh nghiệp hợp lệ; không gồm `ALL`.

`story_duplicate_flag=True` chỉ có nghĩa bài cần được xem xét là tin đăng lại. Notebook không tự động xóa vì việc nhiều báo cùng đăng một câu chuyện có thể phản ánh độ phủ truyền thông.

## 7. Lưu kết quả - tùy chọn

Mặc định `SAVE_OUTPUTS = False` để notebook không tự tạo thêm file. Khi đã kiểm tra kết quả và muốn chạy pipeline chính thức, đổi thành `True`.

In [13]:
# --- Cấu hình lưu ---
SAVE_OUTPUTS = True   # Mặc định lưu để dùng cho các bước sau

# Xác định thư mục gốc dự án
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd

# Lưu vào data/interim (kết quả sau tiền xử lý)
interim_dir = project_root / 'data' / 'interim'
interim_dir.mkdir(parents=True, exist_ok=True)

if SAVE_OUTPUTS:
    for frame in (news_clean, news_for_linking, news_company_for_linking):
        if 'content_hash' in frame.columns:
            frame['content_hash'] = frame['content_hash'].astype('string')
    stock_clean.to_parquet(interim_dir / 'stock_clean.parquet', index=False)
    news_clean.to_parquet(interim_dir / 'news_clean.parquet', index=False)
    news_for_linking.to_parquet(interim_dir / 'news_for_linking.parquet', index=False)
    news_company_for_linking.to_parquet(interim_dir / 'news_company_for_linking.parquet', index=False)
    
    # Nếu có bảng invalid/missing, có thể lưu để tham khảo
    if len(stock_invalid) > 0:
        stock_invalid.to_parquet(interim_dir / 'stock_invalid_rows.parquet', index=False)
    if len(stock_missing_required) > 0:
        stock_missing_required.to_parquet(interim_dir / 'stock_missing_required_rows.parquet', index=False)
    if len(news_missing_required) > 0:
        news_missing_required.to_parquet(interim_dir / 'news_missing_required_rows.parquet', index=False)

    print('Đã lưu kết quả vào:', interim_dir.resolve())
else:
    print('Không lưu file. Dữ liệu sạch đang nằm trong bộ nhớ notebook.')

Đã lưu kết quả vào: D:\KLTN\stock_anomaly_detection\data\interim


## 8. Kết luận Bước 1

Sau khi notebook chạy thành công:

- Dữ liệu cổ phiếu đã sẵn sàng cho bước tạo Daily Return, Absolute Return, Intraday Range, Volume Z-score và Rolling Volatility.
- Dữ liệu tin tức đã được chuẩn hóa mã, thời gian và văn bản.
- Tin `ALL` được giữ cho phân tích ngành nhưng không đưa vào ghép theo công ty.
- Các bài có khả năng đăng lại chỉ được đánh dấu, chưa tự động xóa.
- Chưa căn chỉnh tin sau 15:00 hoặc ngày nghỉ; việc này thuộc bước liên kết tin tức.

**Bước tiếp theo:** tạo `02_feature_engineering.ipynb` và chỉ sử dụng `stock_clean` làm đầu vào.